In [8]:
import os
import numpy as np
import nibabel as nib

def extract_binary_masks(segmentation_path, output_dir=None):
    """
    Extract individual binary masks for each class in a segmentation file.
    
    Parameters:
    -----------
    segmentation_path : str
        Path to the input segmentation NIfTI file (.nii.gz)
    output_dir : str, optional
        Directory where to save the binary masks. If None, masks will be saved
        in the same directory as the input file.
        
    Returns:
    --------
    list
        List of paths to the generated binary mask files
    """
    # Load the segmentation file
    seg_img = nib.load(segmentation_path)
    seg_data = seg_img.get_fdata()
    
    # Get unique class labels, excluding 0 (background)
    unique_labels = np.unique(seg_data)
    unique_labels = unique_labels[unique_labels > 0]
    
    # Create output directory if not provided
    if output_dir is None:
        output_dir = os.path.dirname(segmentation_path)
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Get base filename without extension
    base_filename = os.path.basename(segmentation_path)
    if base_filename.endswith('.nii.gz'):
        base_filename = base_filename[:-7]
    elif base_filename.endswith('.nii'):
        base_filename = base_filename[:-4]
    
    output_paths = []
    
    # Create and save binary mask for each class
    for label in unique_labels:
        # Create binary mask for this class
        binary_mask = np.zeros_like(seg_data)
        binary_mask[np.isclose(seg_data, label)] = 1
        
        # Create a new NIfTI image with the same header as the input
        binary_img = nib.Nifti1Image(binary_mask, seg_img.affine, seg_img.header)
        
        # Save the binary mask with precise label value in filename
        # Use the exact float value to avoid filename conflicts with similar values
        output_path = os.path.join(output_dir, f"{base_filename}_class_{label:.8f}.nii.gz")
        nib.save(binary_img, output_path)
        output_paths.append(output_path)
        
        print(f"Saved binary mask for class {label:.8f} to {output_path}")
    
    return output_paths


segmentation_path = "/data/falcetta/LONDON_TEST/OUT_A2V/TOF_COW_tra_pred.nii.gz"
output_dir = "/data/falcetta/LONDON_TEST/OUT_A2V_VESSELS"

extract_binary_masks(segmentation_path, output_dir)

Saved binary mask for class 1.00001526 to /data/falcetta/LONDON_TEST/OUT_A2V_VESSELS/TOF_COW_tra_pred_class_1.00001526.nii.gz
Saved binary mask for class 2.00000000 to /data/falcetta/LONDON_TEST/OUT_A2V_VESSELS/TOF_COW_tra_pred_class_2.00000000.nii.gz


['/data/falcetta/LONDON_TEST/OUT_A2V_VESSELS/TOF_COW_tra_pred_class_1.00001526.nii.gz',
 '/data/falcetta/LONDON_TEST/OUT_A2V_VESSELS/TOF_COW_tra_pred_class_2.00000000.nii.gz']